# synthetic data
- https://www.kdnuggets.com/5-useful-python-scripts-for-synthetic-data-generation

- While libraries like Faker, SDV, and SynthCity exist — and even large language models (LLMs) are widely used for generating synthetic data — my focus in this article is to avoid relying on these external libraries or AI tools. Instead, you will learn how to achieve the same results by writing your own Python scripts. This provides a better understanding of how to shape a dataset and how biases or errors are introduced. We will start with simple toy scripts to understand the available options. Once you grasp these basics, you can comfortably transition to specialized libraries.
- Faker
- SDV
- SynthCity

## Simple Random

In [9]:
import csv
import random
from datetime import datetime, timedelta
import pandas as pd

In [2]:
random.seed(42)

countries = ["Canada", "UK", "UAE", "Germany", "USA"]
plans = ["Free", "Basic", "Pro", "Enterprise"]

In [3]:
def random_signup_date():
    start = datetime(2024, 1, 1)
    end = datetime(2026, 1, 1)
    delta_days = (end - start).days
    return (start + timedelta(days=random.randint(0, delta_days))).date().isoformat()

In [4]:
rows = []
for i in range(1, 1001):
    age = random.randint(18, 70)
    country = random.choice(countries)
    plan = random.choice(plans)
    monthly_spend = round(random.uniform(0, 500), 2)

    rows.append({
        "customer_id": f"CUST{i:05d}",
        "age": age,
        "country": country,
        "plan": plan,
        "monthly_spend": monthly_spend,
        "signup_date": random_signup_date()
    })

In [7]:
rows[0:2]

[{'customer_id': 'CUST00001',
  'age': 58,
  'country': 'Canada',
  'plan': 'Free',
  'monthly_spend': 370.78,
  'signup_date': '2024-09-07'},
 {'customer_id': 'CUST00002',
  'age': 32,
  'country': 'UK',
  'plan': 'Free',
  'monthly_spend': 338.35,
  'signup_date': '2025-07-12'}]

In [10]:
df = pd.DataFrame(rows)
df.head()

,customer_id,age,country,plan,monthly_spend,signup_date
0,CUST00001,58,Canada,Free,370.78,2024-09-07
1,CUST00002,32,UK,Free,338.35,2025-07-12
2,CUST00003,23,USA,Enterprise,15.89,2024-04-05
3,CUST00004,31,UK,Free,280.62,2025-10-27
4,CUST00005,62,USA,Enterprise,110.22,2025-08-26


In [11]:
df.shape

(1000, 6)

- This script is straightforward: you define fields, choose ranges, and write rows. The random module supports integer generation, floating-point values, random choice, and sampling. The csv module is designed to read and write row-based tabular data. This kind of dataset is suitable for:
    - Frontend demos
    - Dashboard testing
    - API development
    - Learning Structured Query Language (SQL)
    - Unit testing input pipelines
-However, there is a primary weakness to this approach: everything is completely random. This often results in data that looks flat or unnatural. Enterprise customers might spend only 2 dollars, while "Free" users might spend 400. Older users behave exactly like younger ones because there is no underlying structure.

- In real-world scenarios, data rarely behaves this way. Instead of generating values independently, we can introduce relationships and rules. This makes the dataset feel more realistic while remaining fully synthetic. For instance:
    - Enterprise customers should almost never have zero spend
    - Spending ranges should depend on the selected plan
    - Older users might spend slightly more on average
    - Certain plans should be more common than others

# Some control to the data

In [13]:
import csv
import random
random.seed(42)
plans = ["Free", "Basic", "Pro", "Enterprise"]
plans

['Free', 'Basic', 'Pro', 'Enterprise']

In [14]:
def choose_plan():
    roll = random.random()
    if roll < 0.45:
        return "Free"
    if roll < 0.75:
        return "Basic"
    if roll < 0.93:
        return "Pro"
    return "Enterprise"

In [15]:
def generate_spend(age, plan):
    if plan == "Free":
        base = random.uniform(0, 10)
    elif plan == "Basic":
        base = random.uniform(10, 60)
    elif plan == "Pro":
        base = random.uniform(50, 180)
    else:
        base = random.uniform(150, 500)

    if age >= 40:
        base *= 1.15

    return round(base, 2)

rows = []

In [16]:
for i in range(1, 1001):
    age = random.randint(18, 70)
    plan = choose_plan()
    spend = generate_spend(age, plan)

    rows.append({
        "customer_id": f"CUST{i:05d}",
        "age": age,
        "plan": plan,
        "monthly_spend": spend
    })

In [17]:
rows[0:2]

[{'customer_id': 'CUST00001',
  'age': 58,
  'plan': 'Free',
  'monthly_spend': 8.53},
 {'customer_id': 'CUST00002',
  'age': 33,
  'plan': 'Free',
  'monthly_spend': 7.36}]

In [18]:
df2 = pd.DataFrame(rows)
df2.head()

,customer_id,age,plan,monthly_spend
0,CUST00001,58,Free,8.53
1,CUST00002,33,Free,7.36
2,CUST00003,61,Basic,42.86
3,CUST00004,55,Free,0.34
4,CUST00005,31,Free,6.02


- Now the dataset preserves meaningful patterns. Rather than generating random noise, you are simulating behaviors. Effective controls may include:

    - Weighted category selection
    - Realistic minimum and maximum ranges
    - Conditional logic between columns
    - Intentionally added rare edge cases
    - Missing values inserted at low rates
    - Correlated features instead of independent ones

## 2. Simulating Processes for Synthetic Data
 
- Simulation-based generation is one of the best ways to create realistic synthetic datasets. Instead of directly filling columns, you simulate a process. For example, consider a small warehouse where orders arrive, stock decreases, and low stock levels trigger backorders.

In [19]:
import csv
import random
from datetime import datetime, timedelta
random.seed(42)

In [20]:
inventory = {
    "A": 120,
    "B": 80,
    "C": 50
}
inventory

{'A': 120, 'B': 80, 'C': 50}

In [21]:
rows = []
current_time = datetime(2026, 1, 1)

In [22]:
for day in range(30):
    for product in inventory:
        daily_orders = random.randint(0, 12)

        for _ in range(daily_orders):
            qty = random.randint(1, 5)
            before = inventory[product]

            if inventory[product] >= qty:
                inventory[product] -= qty
                status = "fulfilled"
            else:
                status = "backorder"

            rows.append({
                "time": current_time.isoformat(),
                "product": product,
                "qty": qty,
                "stock_before": before,
                "stock_after": inventory[product],
                "status": status
            })

        if inventory[product] < 20:
            restock = random.randint(30, 80)
            inventory[product] += restock
            rows.append({
                "time": current_time.isoformat(),
                "product": product,
                "qty": restock,
                "stock_before": inventory[product] - restock,
                "stock_after": inventory[product],
                "status": "restock"
            })

    current_time += timedelta(days=1)

In [23]:
rows[0:2]

[{'time': '2026-01-01T00:00:00',
  'product': 'A',
  'qty': 1,
  'stock_before': 120,
  'stock_after': 119,
  'status': 'fulfilled'},
 {'time': '2026-01-01T00:00:00',
  'product': 'A',
  'qty': 1,
  'stock_before': 119,
  'stock_after': 118,
  'status': 'fulfilled'}]

In [24]:
df3 = pd.DataFrame(rows)
df3.head()

,time,product,qty,stock_before,stock_after,status
0,2026-01-01T00:00:00,A,1,120,119,fulfilled
1,2026-01-01T00:00:00,A,1,119,118,fulfilled
2,2026-01-01T00:00:00,A,3,118,115,fulfilled
3,2026-01-01T00:00:00,A,2,115,113,fulfilled
4,2026-01-01T00:00:00,A,2,113,111,fulfilled


- This method is excellent because the data is a byproduct of system behavior, which typically yields more realistic relationships than direct random row generation. Other simulation ideas include:
    - Call center queues
    - Ride requests and driver matching
    - Loan applications and approvals
    - Subscriptions and churn
    - Patient appointment flows
    - Website traffic and conversion

## 3. Generating Time Series Synthetic Data
 
- Synthetic data is not just limited to static tables. Many systems produce sequences over time, such as app traffic, sensor readings, orders per hour, or server response times. Here is a simple time series generator for hourly website visits with weekday patterns.

In [25]:
import csv
import random
from datetime import datetime, timedelta
random.seed(42)

In [26]:
start = datetime(2026, 1, 1, 0, 0, 0)
hours = 24 * 30
rows = []

In [27]:
for i in range(hours):
    ts = start + timedelta(hours=i)
    weekday = ts.weekday()

    base = 120
    if weekday >= 5:
        base = 80

    hour = ts.hour
    if 8 <= hour <= 11:
        base += 60
    elif 18 <= hour <= 21:
        base += 40
    elif 0 <= hour <= 5:
        base -= 30

    visits = max(0, int(random.gauss(base, 15)))

    rows.append({
        "timestamp": ts.isoformat(),
        "visits": visits
    })

In [28]:
df4 = pd.DataFrame(rows)
df4.head()

,timestamp,visits
0,2026-01-01T00:00:00,87
1,2026-01-01T01:00:00,87
2,2026-01-01T02:00:00,88
3,2026-01-01T03:00:00,100
4,2026-01-01T04:00:00,88


This approach works well because it incorporates trends, noise, and cyclic behavior while remaining easy to explain and debug.

##  4. Creating Event Logs
- Event logs are another useful script style, ideal for product analytics and workflow testing. Instead of one row per customer, you create one row per action.

In [29]:
import csv
import random
from datetime import datetime, timedelta
random.seed(42)

In [30]:
events = ["signup", "login", "view_page", "add_to_cart", "purchase", "logout"]

rows = []
start = datetime(2026, 1, 1)

In [31]:
for user_id in range(1, 201):
    event_count = random.randint(5, 30)
    current_time = start + timedelta(days=random.randint(0, 10))

    for _ in range(event_count):
        event = random.choice(events)

        if event == "purchase" and random.random() < 0.6:
            value = round(random.uniform(10, 300), 2)
        else:
            value = 0.0

        rows.append({
            "user_id": f"USER{user_id:04d}",
            "event_time": current_time.isoformat(),
            "event_name": event,
            "event_value": value
        })

        current_time += timedelta(minutes=random.randint(1, 180))

In [32]:
df5 = pd.DataFrame(rows)
df5.head()

,user_id,event_time,event_name,event_value
0,USER0001,2026-01-02T00:00:00,signup,0.0
1,USER0001,2026-01-02T01:11:00,login,0.0
2,USER0001,2026-01-02T02:09:00,login,0.0
3,USER0001,2026-01-02T02:36:00,logout,0.0
4,USER0001,2026-01-02T04:56:00,signup,0.0


This format is useful for:

Funnel analysis
Analytics pipeline testing
Business intelligence (BI) dashboards
Session reconstruction
Anomaly detection experiments
A useful technique here is to make events dependent on earlier actions. For example, a purchase should typically follow a login or a page view, making the synthetic log more believable.

## 5. Generating Synthetic Text Data with Templates
 - Synthetic data is also valuable for natural language processing (NLP). You do not always need an LLM to start; you can build effective text datasets using templates and controlled variation. For example, you can create support ticket training data:

In [33]:
import json
import random
random.seed(42)

In [34]:
issues = [
    ("billing", "I was charged twice for my subscription"),
    ("login", "I cannot log into my account"),
    ("shipping", "My order has not arrived yet"),
    ("refund", "I want to request a refund"),
]

tones = ["Please help", "This is urgent", "Can you check this", "I need support"]

records = []

In [35]:
for _ in range(100):
    label, message = random.choice(issues)
    tone = random.choice(tones)

    text = f"{tone}. {message}."
    records.append({
        "text": text,
        "label": label
    })

In [37]:
records[0:3]

[{'text': 'Please help. I was charged twice for my subscription.',
  'label': 'billing'},
 {'text': 'This is urgent. My order has not arrived yet.',
  'label': 'shipping'},
 {'text': 'This is urgent. I cannot log into my account.', 'label': 'login'}]

In [38]:
df6 = pd.DataFrame(records)
df6.head()

,text,label
0,Please help. I was charged twice for my subscr...,billing
1,This is urgent. My order has not arrived yet.,shipping
2,This is urgent. I cannot log into my account.,login
3,Please help. I was charged twice for my subscr...,billing
4,Please help. I want to request a refund.,refund


This approach works well for:

Text classification demos
Intent detection
Chatbot testing
Prompt evaluation

# Final Thoughts
Synthetic data scripts are powerful tools, but they can be implemented incorrectly. Be sure to avoid these common mistakes:

Making all values uniformly random
Forgetting dependencies between fields
Generating values that violate business logic
Assuming synthetic data is inherently safe by default
Creating data that is too "clean" to be useful for testing real-world edge cases
Using the same pattern so frequently that the dataset becomes predictable and unrealistic